<a href="https://colab.research.google.com/github/Ayanc7524/IPL-Prediction-Model1/blob/main/IPL_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Full code for a basic IPL prediction system with support for:
# - Model training
# - Prediction updates
# - Handling injuries/weather
# - Retraining pipeline
# - Player trend visualization (basic plotly)

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
import plotly.express as px
import os

In [3]:
def feature_engineering(df):
    # Example features
    df['team_form_diff'] = df['team1_form'] - df['team2_form']
    df['venue_score_bias'] = df['venue_avg_score'] - df['league_avg_score']
    df['injury_impact'] = df['key_player_out'].apply(lambda x: 1 if x else 0)
    df['weather_disruption'] = df['rain_chance'].apply(lambda x: 1 if x > 0.5 else 0)
    return df[['team_form_diff', 'venue_score_bias', 'injury_impact', 'weather_disruption']], df['match_winner']

In [4]:
def train_model(data_path):
    df = pd.read_csv(data_path)
    X, y = feature_engineering(df)
    model = GradientBoostingClassifier()
    model.fit(X, y)
    joblib.dump(model, 'ipl_match_predictor.pkl')
    return model

In [5]:
def update_live_prediction(ball_data_df):
    # Extract live features
    run_rate = ball_data_df['total_runs'].sum() / (ball_data_df['ball'].count() / 6)
    wickets = ball_data_df['is_wicket'].sum()
    return {'run_rate': run_rate, 'wickets_lost': wickets}

In [6]:
def predict_outcome(input_dict):
    model = joblib.load('ipl_match_predictor.pkl')
    input_df = pd.DataFrame([input_dict])
    X, _ = feature_engineering(input_df)
    pred = model.predict(X)
    prob = model.predict_proba(X)
    return {'prediction': pred[0], 'confidence': prob.max()}

In [7]:
def retrain_pipeline(new_data_path):
    print("Retraining model with new data...")
    return train_model(new_data_path)

In [8]:
def plot_player_trend(df, player_name):
    player_df = df[df['player'] == player_name].sort_values('match_date')
    fig = px.line(player_df, x='match_date', y='runs', title=f"{player_name} Runs Trend")
    fig.show()

In [10]:
if __name__ == '__main__':
    # Step 1: Train model with historical data
    if not os.path.exists('ipl_match_predictor.pkl'):
        train_model('ipl_historical_data.csv')

In [13]:
sample_input = {
        'team1_form': 0.8,
        'team2_form': 0.6,
        'venue_avg_score': 170,
        'league_avg_score': 160,
        'key_player_out': True,
        'rain_chance': 0.7,
        'match_winner': 1
}
print(predict_outcome(sample_input))

{'prediction': np.int64(0), 'confidence': np.float64(0.8207205683775853)}


In [15]:
live_data = pd.DataFrame({
        'ball': range(1, 37),
        'total_runs': np.random.randint(0, 7, size=36),
        'is_wicket': np.random.choice([0, 1], size=36, p=[0.9, 0.1])
    })
print(update_live_prediction(live_data))

{'run_rate': np.float64(20.166666666666668), 'wickets_lost': np.int64(1)}


In [18]:
dummy_perf_data = pd.DataFrame({
        'player': ['Virat Kohli'] * 5,
        'match_date': pd.date_range(end=pd.Timestamp.today(), periods=5),
        'runs': [45, 32, 76, 20, 58]
    })
plot_player_trend(dummy_perf_data, 'Virat Kohli')
